In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [328]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)


def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=-1)
    #constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f+extra)
    print(c+extra)
    return(prob.value - (1-np.sum(a))*r_f <= c +1e-6)

In [233]:
def cutting_plane(R,r,c,p,m,r_f,sets):
    nonstop = True
    iterations = 1
    while nonstop == True:
        [a,obj] = solvenominal(sets,p,R,r,m,r_f,c)
        print(obj)
        newrank = np.argsort(R.dot(a))
        [sets,added] = makesetflex(sets, newrank)
        if robustcheck(a,R,r,c,p,m,r_f) == True:
            return(a,obj,iterations)
        iterations = iterations + 1
    

In [353]:
def riskcalc(a,R,p,alfa,r_f):
    x = R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.max(x) > 0:
        extra = np.max(x)
        x = x - np.max(x)
    x = -x
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk - extra
    print("the nominal risk of a:", risk)

    
def norisksolve(p,R,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [-1<=a, a<=1]
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [264]:
np.random.seed(5)

In [386]:
N=30
p = (np.zeros(N)+1)*1/N
I = 1
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.05515823]
[[-0.2456747 ]
 [ 0.01317014]
 [-0.11545419]
 [ 0.04231286]
 [-0.09390811]
 [-0.014924  ]
 [-0.07154646]
 [ 0.02552909]
 [ 0.16500069]
 [ 0.0986919 ]
 [ 0.04981658]
 [ 0.25448735]
 [-0.22236318]
 [ 0.08330546]
 [ 0.28395031]
 [ 0.16189604]
 [ 0.2749394 ]
 [-0.13525289]
 [ 0.27919439]
 [-0.09022493]
 [ 0.27603427]
 [ 0.10085268]
 [ 0.22195395]
 [-0.10615044]
 [ 0.00698267]
 [-0.31775169]
 [ 0.12298171]
 [ 0.19989352]
 [ 0.12700421]
 [ 0.28000023]]


In [389]:
r = 10
m = 0.95    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.02
sets = [[0]]
cut_result = cutting_plane(R,r,c,p,m,r_f,sets)
print(cut_result)


0.004568052567449021
0.019999999962342585
0.019999999999999997
(array([0.065882]), 0.004568052567449021, 1)


In [366]:
sets

[[0],
 [8],
 [8, 4],
 [8, 4, 3],
 [8, 4, 3, 2],
 [8, 4, 3, 2, 7],
 [8, 4, 3, 2, 7, 5],
 [8, 4, 3, 2, 7, 5, 0],
 [8, 4, 3, 2, 7, 5, 0, 6],
 [8, 4, 3, 2, 7, 5, 0, 6, 1],
 [8, 4, 3, 2, 7, 5, 0, 6, 1, 9]]

In [352]:
riskcalc(cut_result[0],R,p,m,r_f)   

the nominal risk of a: 0.019345254473104367


In [359]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
resultall = solvenominal (psets,p,R,r,m,r_f,c)
print(resultall)
print(np.argsort(R.dot(resultall[0])))

(array([0.41410879, 0.57722393, 0.66342105]), 0.11875222635544834)
[1 5 0 3 2 4]


In [354]:
norisksolve(p,R,r_f)

(array([1., 1., 1.]), 0.22139611196311468)

In [249]:
a = norisksolve(p,R,r_f)[0]
riskcalc(a,R,p,m,r_f)

the nominal risk of a: 0.21526622824823638


In [380]:
def phi_div(p,q,r):
    phi_cons = 0
    for i in range(len(p)):
        phi_cons = q[i]*np.log(q[i]/p[i])+phi_cons
    print(phi_cons <= r)
    print(phi_cons)

In [385]:
q = np.zeros(N)+0.0001
q[0]=1-sum(q[1:len(q)])
phi_div(p,q,10)

True
3.371591603654162


array([-0.03078842])

array([1, 2, 4, 3, 0], dtype=int64)

10.3

0.3508771929824561